In [2]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import pyarrow.parquet as pq
import numpy as np
import faiss

table = pq.read_table("../data/processed/product_documents.parquet")
docs = table.to_pandas()

docs.head()

c:\Users\rocco\miniforge3\envs\amazon-recommender\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,parent_asin,product_title,document
0,B0B9J2JWSX,ASUS TUF Gaming 23.8” 1080P Monitor (VG249Q1R)...,Computers ASUS TUF Gaming 23.8” 1080P Monitor ...
1,B0C2PVFRWV,"USB Type C Cable, Anker [2-Pack 6Ft] Premium N...","Cell Phones & Accessories USB Type C Cable, An..."
2,B01KIOU4EO,Echo Show - 1st Generation Black,Amazon Devices Echo Show - 1st Generation Blac...
3,B07GBYKWGW,ASUS AC1200 Dual Band WiFi Repeater & Range Ex...,Computers ASUS AC1200 Dual Band WiFi Repeater ...
4,B00O5VX2KK,"ASUS C300 ChromeBook 13.3 Inch (Intel Celeron,...",Computers ASUS C300 ChromeBook 13.3 Inch (Inte...


In [3]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2203.44it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
# Generate embeddings for documents
sentences = docs["document"].to_list()
embeddings = model.encode(sentences)
print(embeddings.shape)

(500, 384)


In [5]:
# Step 1: Build the FAISS index
embeddings_array = np.array(embeddings).astype('float32')
dimension = embeddings_array.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embeddings_array)

# Step 2: Query function
def search(query, k=5):
    query_embedding = model.encode([query]).astype('float32')
    distances, indices = index.search(query_embedding, k)
    
    results = []
    for dist, idx in zip(distances[0], indices[0]):
        results.append({
            'parent_asin': docs.iloc[idx]['parent_asin'],
            'product_title': docs.iloc[idx]['product_title'],
            'distance': dist
        })
    return results

# Step 3: Query
results = search("sony headphones")
for r in results:
    print(r['product_title'])
    print(f"Distance: {r['distance']:.4f}")
    print("---")

Avantree ANC032 Active Noise Cancelling Headphones Over Ear with Microphone for Home Office, Confere
Distance: 0.9143
---
Audeze LCD-X Over Ear Open Back Headphone with New Suspension Headband and Travel case
Distance: 0.9193
---
Avantree TR302 Two Way 3.5mm Dual Headphone Jack Splitter, AUX Stereo Earphone Earbuds Y Audio Split
Distance: 0.9739
---
WELOYA Golden Plated Airline Airplane Flight Adapter (Pack 0f 6)
Distance: 0.9907
---
DOBOPO Wireless Earbuds Bluetooth 5.3 Headphones 50Hrs Playtime Sports Earphones Over-Ear Earhooks H
Distance: 1.0263
---


In [6]:
# faiss.write_index(index, "../data/processed/semantic_search_index.faiss")

In [10]:
results_df = pd.DataFrame(results)
results_df

,parent_asin,product_title,distance
0,B079GMZHT7,Avantree ANC032 Active Noise Cancelling Headph...,0.914349
1,B00QD0TO30,Audeze LCD-X Over Ear Open Back Headphone with...,0.919325
2,B09MCTFR69,Avantree TR302 Two Way 3.5mm Dual Headphone Ja...,0.973904
3,B06Y2TYFJ8,WELOYA Golden Plated Airline Airplane Flight A...,0.990706
4,B0BV951H9Z,DOBOPO Wireless Earbuds Bluetooth 5.3 Headphon...,1.026347
